# Bank Marketing Campaign - Decision Tree Classifier

This notebook demonstrates the process of building a Decision Tree Classifier to predict whether a customer will subscribe to a term deposit based on demographic and behavioral data.

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

## 2. Load and Clean Dataset

We use the enriched `bank-additional-full.csv` dataset, which contains 41,188 instances and 21 features.

In [ ]:
data_path = os.path.join("bank-additional", "bank-additional-full.csv")
df = pd.read_csv(data_path, sep=";")
print(f"Data loaded. Shape: {df.shape}")
df.head()

### Target Leakage Prevention
As noted in the dataset documentation, the `duration` column (contact duration in seconds) is not known before a call is made, and after the call, the target variable `y` is known. Therefore, we discard `duration` to construct a realistic predictive model.

In [ ]:
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
    print("Dropped 'duration' to prevent target leakage.")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable Distribution
sns.countplot(x='y', data=df, hue='y', palette='viridis', legend=False)
plt.title("Subscription Rates (y)")
plt.show()

In [ ]:
# Age Distribution by Subscription
sns.histplot(x='age', hue='y', data=df, kde=True, multiple='stack', palette='viridis')
plt.title("Age Distribution by Subscription status")
plt.show()

In [ ]:
# Job Type vs Subscription
sns.countplot(y='job', hue='y', data=df, palette='viridis')
plt.title("Job Types vs Subscriptions")
plt.tight_layout()
plt.show()

## 4. Preprocessing & Encoding

In [ ]:
# Convert target to binary
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Categorical encoding
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols)

X = df_encoded.drop(columns=['y'])
y = df_encoded['y']

print(f"Encoded features count: {X.shape[1]}")

## 5. Model Training & Tuning

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Hyperparameter search using GridSearchCV
param_grid = {
    'max_depth': [3, 4, 5, 6, 8, 10],
    'min_samples_leaf': [10, 20, 50, 100],
    'criterion': ['entropy', 'gini']
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_clf = grid.best_estimator_
print(f"Best Parameters: {grid.best_params_}")

## 6. Evaluation

In [ ]:
y_pred = best_clf.predict(X_test)
y_prob = best_clf.predict_proba(X_test)[:, 1]

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Feature Importance
importances = best_clf.feature_importances_
indices = np.argsort(importances)[::-1]
top_k = 15

plt.title("Top Feature Importances")
sns.barplot(x=importances[indices[:top_k]], y=np.array(X.columns)[indices[:top_k]], palette='viridis', hue=np.array(X.columns)[indices[:top_k]], legend=False)
plt.xlabel('Relative Importance')
plt.show()

## 7. Plotting the Decision Tree

Let's visualize the top splits of the decision tree classifier.

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(best_clf, max_depth=3, feature_names=list(X.columns), class_names=['no', 'yes'], filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Structure (Max Depth 3 displayed)")
plt.show()